# 🔬 VQA-E: Visual Question Answering with Explanation

## Dự án: Xây dựng hệ thống VQA sinh câu trả lời kèm giải thích

**Mục tiêu**: Cho 1 ảnh + 1 câu hỏi → sinh ra `"answer because explanation"`

**Kiến trúc**: CNN Encoder + BiLSTM Question Encoder → Gated Fusion → LSTM Decoder (có/không Attention)

### 4 Model variants:
| Model | CNN | Attention | Mô tả |
|-------|-----|-----------|--------|
| **A** | SimpleCNN (scratch) | ❌ | Baseline đơn giản |
| **B** | ResNet101 (pretrained) | ❌ | Pretrained CNN, no attention |
| **C** | SimpleCNN Spatial | ✅ Dual Attention | Scratch CNN + Attention |
| **D** | ResNet101 Spatial | ✅ Dual Attention | Full model (best) |

### 11 Cải tiến kiến trúc:
| # | Cải tiến | Mô tả |
|---|----------|--------|
| 1 | Label Smoothing | `CrossEntropyLoss(label_smoothing=0.1)` — giảm overconfident |
| 2 | BiLSTM Encoder | Bidirectional LSTM cho question — bắt context cả 2 chiều |
| 3 | Gated Fusion | `gate * tanh(W_img) + (1-gate) * tanh(W_q)` thay Hadamard |
| 4 | Dual Attention | Attend cả image regions VÀ question tokens |
| 5 | LR Warmup | 3 epochs warmup (lr/10 → lr) trước ReduceLROnPlateau |
| 6 | GloVe 300d | Pretrained word embeddings (300d + projection → 512d) |
| 7 | Weight Tying | `fc.weight = embedding.weight` — chia sẻ embedding matrix |
| 8 | BERTScore | Semantic similarity metric bằng BERT embeddings |
| 9 | Coverage | Phạt decoder khi attend lại vùng đã attend |
| 10 | N-gram Blocking | Block repeated trigrams trong beam search |
| 11 | Grad Accumulation | Effective batch = batch_size × accum_steps |

### Training Plan (3 phases):
- **Phase 1**: 10 epochs — baseline comparison (all 4 models)
- **Phase 2**: +5 epochs — fine-tune CNN (B/D) + Scheduled Sampling
- **Phase 3**: +5 epochs — full optimization (coverage + GloVe + augmentation)

---
## 1. Setup & Environment

### 1.1 Mount Google Drive & Clone Repository

In [ ]:
# Mount Google Drive (chạy trên Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repository từ GitHub (hoặc copy từ Drive)
import os

# Option 1: Clone từ GitHub
# !git clone https://github.com/YOUR_USERNAME/vqa_new.git /content/vqa_new

# Option 2: Copy từ Google Drive
PROJECT_DIR = '/content/vqa_new'
DRIVE_DIR = '/content/drive/MyDrive/vqa_new'

if os.path.exists(DRIVE_DIR):
    !cp -r {DRIVE_DIR} {PROJECT_DIR}
    print(f"Copied project from Drive to {PROJECT_DIR}")
else:
    print(f"Drive directory not found: {DRIVE_DIR}")
    print("Please clone from GitHub or upload manually.")

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

### 1.2 Kiểm tra GPU & Cài đặt dependencies

In [ ]:
# Kiểm tra GPU
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    cap = torch.cuda.get_device_capability()
    print(f"Compute cap     : {cap[0]}.{cap[1]}")
    print(f"BFloat16 support: {'Yes' if cap[0] >= 8 else 'No'}")

In [ ]:
# Cài đặt dependencies
!pip install -q nltk bert-score tqdm pillow matplotlib

# Download NLTK data cho METEOR metric
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("Dependencies installed!")

In [ ]:
# Kiểm tra cấu trúc project
!find . -name "*.py" -not -path "./__pycache__/*" -not -path "./*/__pycache__/*" | sort
print()
!ls -la data/raw/ 2>/dev/null || echo "No data directory yet" 

---
## 2. Data Preparation

### 2.1 Download COCO 2014 Images

VQA-E sử dụng ảnh từ COCO 2014 (giống VQA 2.0). Cần download:
- `train2014` (~13 GB, 82,783 images) — dùng để train
- `val2014` (~6 GB, 40,504 images) — dùng để validate

In [ ]:
# Download COCO 2014 images (chạy trên Colab)
import os

os.makedirs('data/raw/images', exist_ok=True)

# === Train2014 ===
if not os.path.exists('data/raw/images/train2014') or len(os.listdir('data/raw/images/train2014')) < 1000:
    print("Downloading COCO train2014 images (~13 GB)...")
    !wget -q http://images.cocodataset.org/zips/train2014.zip -O data/raw/images/train2014.zip
    !unzip -q data/raw/images/train2014.zip -d data/raw/images/
    !rm data/raw/images/train2014.zip
    print(f"Train images: {len(os.listdir('data/raw/images/train2014'))}")
else:
    print(f"Train images already exist: {len(os.listdir('data/raw/images/train2014'))}")

# === Val2014 ===
if not os.path.exists('data/raw/images/val2014') or len(os.listdir('data/raw/images/val2014')) < 1000:
    print("Downloading COCO val2014 images (~6 GB)...")
    !wget -q http://images.cocodataset.org/zips/val2014.zip -O data/raw/images/val2014.zip
    !unzip -q data/raw/images/val2014.zip -d data/raw/images/
    !rm data/raw/images/val2014.zip
    print(f"Val images: {len(os.listdir('data/raw/images/val2014'))}")
else:
    print(f"Val images already exist: {len(os.listdir('data/raw/images/val2014'))}")

### 2.2 Kiểm tra VQA-E Annotation Files

VQA-E data gồm 2 file JSON:
- `VQA-E_train_set.json` (181,298 samples)
- `VQA-E_val_set.json` (88,488 samples)

Mỗi entry chứa: `question`, `multiple_choice_answer`, `explanation`, `img_id`

In [ ]:
# Kiểm tra VQA-E data
import json

os.makedirs('data/raw/vqa_e_json', exist_ok=True)

for split in ['train', 'val']:
    path = f'data/raw/vqa_e_json/VQA-E_{split}_set.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f"{split}: {len(data)} annotations")
        print(f"  Keys: {list(data[0].keys())}")
        print(f"  Sample Q: {data[0]['question']}")
        print(f"  Sample A: {data[0]['multiple_choice_answer']}")
        exp = data[0].get('explanation', [''])[0]
        print(f"  Sample E: {exp}")
        print()
    else:
        print(f"⚠️  File not found: {path}")
        print(f"  Please upload VQA-E data to data/raw/vqa_e_json/")

---
## 3. Data Exploration & Visualization

### 3.1 Thống kê dataset

In [ ]:
import json
import numpy as np
from collections import Counter

with open('data/raw/vqa_e_json/VQA-E_train_set.json') as f:
    train_data = json.load(f)

# Thống kê cơ bản
print(f"{'='*50}")
print(f"VQA-E Train Set Statistics")
print(f"{'='*50}")
print(f"Total annotations : {len(train_data)}")

# Unique images
img_ids = set(ann['img_id'] for ann in train_data)
print(f"Unique images     : {len(img_ids)}")
print(f"Avg Q/image       : {len(train_data)/len(img_ids):.1f}")

# Question types
q_types = Counter(ann.get('question_type', 'unknown') for ann in train_data)
print(f"\nTop 10 question types:")
for qt, count in q_types.most_common(10):
    print(f"  {qt:<25} {count:>6} ({count/len(train_data)*100:.1f}%)")

# Answer types
a_types = Counter(ann.get('answer_type', 'unknown') for ann in train_data)
print(f"\nAnswer types:")
for at, count in a_types.most_common():
    print(f"  {at:<25} {count:>6} ({count/len(train_data)*100:.1f}%)")

In [ ]:
# Phân bố độ dài câu trả lời + explanation
import re

def tokenize(text):
    return re.findall(r'\w+', text.lower())

q_lengths = []
a_lengths = []  # answer only
ae_lengths = []  # answer + because + explanation

for ann in train_data:
    q_lengths.append(len(tokenize(ann['question'])))
    a_lengths.append(len(tokenize(ann['multiple_choice_answer'])))
    
    exp = ann.get('explanation', [''])[0] if ann.get('explanation') else ''
    if exp:
        full = f"{ann['multiple_choice_answer']} because {exp}"
    else:
        full = ann['multiple_choice_answer']
    ae_lengths.append(len(tokenize(full)))

print(f"Question length  : mean={np.mean(q_lengths):.1f}, median={np.median(q_lengths):.0f}, max={max(q_lengths)}")
print(f"Answer length    : mean={np.mean(a_lengths):.1f}, median={np.median(a_lengths):.0f}, max={max(a_lengths)}")
print(f"Answer+Exp length: mean={np.mean(ae_lengths):.1f}, median={np.median(ae_lengths):.0f}, max={max(ae_lengths)}")

### 3.2 Visualize Samples

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

# Hiển thị 6 mẫu ngẫu nhiên
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
samples = random.sample(train_data, 6)

for ax, sample in zip(axes.flatten(), samples):
    img_id = sample['img_id']
    img_path = f"data/raw/images/train2014/COCO_train2014_{img_id:012d}.jpg"
    
    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, 'Image\nnot found', ha='center', va='center', fontsize=12)
    
    q = sample['question']
    a = sample['multiple_choice_answer']
    exp = sample.get('explanation', [''])[0][:60]
    
    ax.set_title(f"Q: {q}\nA: {a}\nE: {exp}...", fontsize=9, wrap=True)
    ax.axis('off')

plt.tight_layout()
plt.savefig('results/data_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/data_samples.png")

In [ ]:
# Phân bố histogram
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(q_lengths, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Question Length Distribution')
axes[0].set_xlabel('Number of tokens')
axes[0].axvline(np.mean(q_lengths), color='red', linestyle='--', label=f'mean={np.mean(q_lengths):.1f}')
axes[0].legend()

axes[1].hist(a_lengths, bins=20, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Answer Length Distribution')
axes[1].set_xlabel('Number of tokens')
axes[1].axvline(np.mean(a_lengths), color='red', linestyle='--', label=f'mean={np.mean(a_lengths):.1f}')
axes[1].legend()

axes[2].hist(ae_lengths, bins=50, color='seagreen', edgecolor='black', alpha=0.7)
axes[2].set_title('Answer+Explanation Length Distribution')
axes[2].set_xlabel('Number of tokens')
axes[2].axvline(np.mean(ae_lengths), color='red', linestyle='--', label=f'mean={np.mean(ae_lengths):.1f}')
axes[2].legend()

plt.tight_layout()
plt.savefig('results/length_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Build Vocabulary

Xây dựng 2 vocabulary riêng biệt:
- **Question Vocab**: từ tất cả câu hỏi (threshold=3)
- **Answer Vocab**: từ "answer because explanation" (threshold=3)

Special tokens: `<pad>` (0), `<start>` (1), `<end>` (2), `<unk>` (3)

In [ ]:
# Build vocabulary từ VQA-E data
!python src/scripts/1_build_vocab.py

In [ ]:
# Verify vocabulary
import sys
sys.path.append('src')
from vocab import Vocabulary

vocab_q = Vocabulary(); vocab_q.load('data/processed/vocab_questions.json')
vocab_a = Vocabulary(); vocab_a.load('data/processed/vocab_answers.json')

print(f"Question vocab : {len(vocab_q)} tokens")
print(f"Answer vocab   : {len(vocab_a)} tokens")
print(f"\nSpecial tokens:")
for tok in ['<pad>', '<start>', '<end>', '<unk>']:
    print(f"  {tok:10s} → idx {vocab_q.word2idx[tok]}")

# Test numericalize
q_test = "What color is the cat?"
a_test = "black because the cat in the image appears to be black"
print(f"\nQ: '{q_test}'")
print(f"   → {vocab_q.numericalize(q_test)}")
print(f"A: '{a_test}'")
print(f"   → {vocab_a.numericalize(a_test)[:15]}...")

---
## 5. Model Architecture Overview

### Pipeline chung:

```
Image (224×224×3)
    │
    ▼
CNN Encoder ──────────────────┐
    │                         │
    │ Model A/B: (B, H)       │ Model C/D: (B, 49, H)
    │ global vector            │ spatial features
    │                         │
    ▼                         ▼
Question ──→ BiLSTM ──→ q_feature (B, H)
    │                   q_hidden (B, qlen, H)
    │                         │
    ▼                         │
Gated Fusion ←────────────────┘
    │
    ▼
h_0 = fusion.repeat(num_layers)
c_0 = zeros
    │
    ▼
LSTM Decoder
    │
    ├── Model A/B: Teacher Forcing only
    │   input = embed(token)
    │
    └── Model C/D: Dual Attention
        At each step:
        1. img_context = BahdanauAttn(h, img_features)  ← coverage
        2. q_context   = BahdanauAttn(h, q_hidden)
        3. lstm_input = [embed; img_context; q_context]
    │
    ▼
logits (B, seq_len, vocab_size)
```

### Key dimensions:
- `embed_size = 512` (GloVe 300d + projection khi dùng GloVe)
- `hidden_size = 1024`
- `num_layers = 2`
- `attn_dim = 512`

In [ ]:
# Xem tổng quan kiến trúc model
import sys
sys.path.append('src')
from models.vqa_models import VQAModelA, VQAModelB, VQAModelC, VQAModelD

# Instantiate all 4 models
models_info = {}
for name, ModelClass in [('A', VQAModelA), ('B', VQAModelB), ('C', VQAModelC), ('D', VQAModelD)]:
    m = ModelClass(vocab_size=len(vocab_q), answer_vocab_size=len(vocab_a))
    n_params = sum(p.numel() for p in m.parameters())
    n_trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    models_info[name] = {'total': n_params, 'trainable': n_trainable}
    print(f"Model {name}: {n_params:>12,} total | {n_trainable:>12,} trainable")
    del m

print(f"\nQ-vocab: {len(vocab_q)} | A-vocab: {len(vocab_a)}")

In [ ]:
# Test forward pass cho tất cả models
import torch
import torch.nn.functional as F

B = 2
imgs = torch.randn(B, 3, 224, 224)
qs = torch.randint(0, len(vocab_q), (B, 15))
ans = torch.randint(0, len(vocab_a), (B, 20))

print("Forward pass test:")
for name, ModelClass in [('A', VQAModelA), ('B', VQAModelB), ('C', VQAModelC), ('D', VQAModelD)]:
    m = ModelClass(vocab_size=len(vocab_q), answer_vocab_size=len(vocab_a))
    result = m(imgs, qs, ans[:, :-1])
    if isinstance(result, tuple):
        logits, cov_loss = result
        print(f"  Model {name}: logits={logits.shape}, coverage_loss={cov_loss.item():.4f}")
    else:
        print(f"  Model {name}: logits={result.shape}")
    del m

print("All models OK! ✅")

---
## 6. Phase 1: Baseline Training (10 epochs)

Train tất cả 4 models với cấu hình baseline:
- 10 epochs
- LR = 1e-3 (with 3-epoch warmup)
- Batch size = 128
- Label smoothing = 0.1
- Weight decay = 1e-5
- No fine-tuning CNN (ResNet frozen)
- No scheduled sampling
- No coverage, no GloVe

**Lưu ý**: Trên Colab T4 (16GB VRAM), có thể cần giảm batch_size xuống 64.

In [ ]:
# ═══ PHASE 1: Train Model A (SimpleCNN, no attention) ═══
!python src/train.py \
    --model A \
    --epochs 10 \
    --lr 1e-3 \
    --batch_size 128 \
    --num_workers 4

In [ ]:
# ═══ PHASE 1: Train Model B (ResNet101, no attention) ═══
!python src/train.py \
    --model B \
    --epochs 10 \
    --lr 1e-3 \
    --batch_size 128 \
    --num_workers 4

In [ ]:
# ═══ PHASE 1: Train Model C (SimpleCNN + Dual Attention) ═══
!python src/train.py \
    --model C \
    --epochs 10 \
    --lr 1e-3 \
    --batch_size 64 \
    --num_workers 4

In [ ]:
# ═══ PHASE 1: Train Model D (ResNet101 + Dual Attention) ═══
!python src/train.py \
    --model D \
    --epochs 10 \
    --lr 1e-3 \
    --batch_size 64 \
    --num_workers 4

---
## 7. Phase 1: Evaluation & Comparison

### 7.1 Training Curves

In [ ]:
# Plot training curves cho tất cả models
import json
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'A': '#1f77b4', 'B': '#ff7f0e', 'C': '#2ca02c', 'D': '#d62728'}

for model_type in ['A', 'B', 'C', 'D']:
    history_path = f'checkpoints/history_model_{model_type.lower()}.json'
    if os.path.exists(history_path):
        with open(history_path) as f:
            history = json.load(f)
        epochs = range(1, len(history['train_loss']) + 1)
        axes[0].plot(epochs, history['train_loss'], label=f'Model {model_type}',
                     color=colors[model_type], linewidth=2)
        axes[1].plot(epochs, history['val_loss'], label=f'Model {model_type}',
                     color=colors[model_type], linewidth=2)

axes[0].set_title('Training Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Loss', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Phase 1: Baseline Training (10 epochs)', fontsize=16, fontweight='bold')
plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/phase1_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 So sánh metrics (BLEU-4, METEOR, BERTScore)

In [ ]:
# So sánh tất cả 4 models trên validation set
# Sử dụng greedy decode (nhanh hơn beam search)
!python src/compare.py --epoch 10 --models A,B,C,D

In [ ]:
# Đánh giá chi tiết Model D (best expected) với beam search
!python src/evaluate.py --model_type D --checkpoint checkpoints/model_d_epoch10.pth --beam_width 5

### 7.3 Sample Predictions

In [ ]:
# Xem một số predictions từ mỗi model
import sys
sys.path.append('src')
import torch
from PIL import Image
from torchvision import transforms
from vocab import Vocabulary
from inference import get_model, greedy_decode, greedy_decode_with_attention

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

vocab_q = Vocabulary(); vocab_q.load('data/processed/vocab_questions.json')
vocab_a = Vocabulary(); vocab_a.load('data/processed/vocab_answers.json')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load val samples
with open('data/raw/vqa_e_json/VQA-E_val_set.json') as f:
    val_data = json.load(f)

# Show predictions cho 5 samples
import random
samples = random.sample(val_data, 5)

for model_type in ['A', 'D']:
    ckpt = f'checkpoints/model_{model_type.lower()}_epoch10.pth'
    if not os.path.exists(ckpt):
        print(f"Checkpoint not found: {ckpt}, skipping Model {model_type}")
        continue
    
    model = get_model(model_type, len(vocab_q), len(vocab_a))
    model.load_state_dict(torch.load(ckpt, map_location='cpu'))
    model.to(DEVICE).eval()
    
    print(f"\n{'='*60}")
    print(f"Model {model_type} Predictions")
    print(f"{'='*60}")
    
    for s in samples:
        img_path = f"data/raw/images/val2014/COCO_val2014_{s['img_id']:012d}.jpg"
        if not os.path.exists(img_path):
            continue
        
        img_tensor = transform(Image.open(img_path).convert('RGB'))
        q_tensor = torch.tensor(vocab_q.numericalize(s['question']), dtype=torch.long)
        
        if model_type in ('C', 'D'):
            pred = greedy_decode_with_attention(model, img_tensor, q_tensor, vocab_a, device=DEVICE)
        else:
            pred = greedy_decode(model, img_tensor, q_tensor, vocab_a, device=DEVICE)
        
        gt = s['multiple_choice_answer']
        exp = s.get('explanation', [''])[0][:80]
        print(f"  Q: {s['question']}")
        print(f"  GT: {gt} because {exp}...")
        print(f"  Pred: {pred}")
        print()
    
    del model

---
## 8. Phase 2: Fine-tune + Scheduled Sampling (+5 epochs)

Phase 2 thêm các kỹ thuật training nâng cao:
- **Scheduled Sampling** (k=5): giảm exposure bias — dần thay GT token bằng prediction
- **CNN Fine-tuning** (Model B/D): unfreeze ResNet layer3+layer4 với LR nhỏ hơn (×0.1)
- **Data Augmentation**: RandomHorizontalFlip + ColorJitter
- Resume từ Phase 1 checkpoint

In [ ]:
# ═══ PHASE 2: Model A (+5 epochs, scheduled sampling + augment) ═══
!python src/train.py \
    --model A \
    --epochs 5 \
    --lr 5e-4 \
    --batch_size 128 \
    --scheduled_sampling \
    --augment \
    --early_stopping 3 \
    --resume checkpoints/model_a_resume.pth

In [ ]:
# ═══ PHASE 2: Model B (+5 epochs, finetune CNN + SS + augment) ═══
!python src/train.py \
    --model B \
    --epochs 5 \
    --lr 5e-4 \
    --batch_size 128 \
    --scheduled_sampling \
    --finetune_cnn \
    --cnn_lr_factor 0.1 \
    --augment \
    --early_stopping 3 \
    --resume checkpoints/model_b_resume.pth

In [ ]:
# ═══ PHASE 2: Model C (+5 epochs, SS + augment) ═══
!python src/train.py \
    --model C \
    --epochs 5 \
    --lr 5e-4 \
    --batch_size 64 \
    --scheduled_sampling \
    --augment \
    --early_stopping 3 \
    --resume checkpoints/model_c_resume.pth

In [ ]:
# ═══ PHASE 2: Model D (+5 epochs, finetune CNN + SS + augment) ═══
!python src/train.py \
    --model D \
    --epochs 5 \
    --lr 5e-4 \
    --batch_size 64 \
    --scheduled_sampling \
    --finetune_cnn \
    --cnn_lr_factor 0.1 \
    --augment \
    --early_stopping 3 \
    --resume checkpoints/model_d_resume.pth

---
## 9. Phase 2: Evaluation

In [ ]:
# Plot updated training curves (Phase 1 + Phase 2)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_type in ['A', 'B', 'C', 'D']:
    history_path = f'checkpoints/history_model_{model_type.lower()}.json'
    if os.path.exists(history_path):
        with open(history_path) as f:
            history = json.load(f)
        epochs = range(1, len(history['train_loss']) + 1)
        axes[0].plot(epochs, history['train_loss'], label=f'Model {model_type}',
                     color=colors[model_type], linewidth=2)
        axes[1].plot(epochs, history['val_loss'], label=f'Model {model_type}',
                     color=colors[model_type], linewidth=2)

# Phase boundary
for ax in axes:
    ax.axvline(10, color='gray', linestyle='--', alpha=0.5, label='Phase 1→2')
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[1].set_title('Validation Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')

plt.suptitle('Phase 1+2 Training Curves', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/phase2_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# So sánh Phase 2 (epoch 15)
!python src/compare.py --epoch 15 --models A,B,C,D

In [ ]:
# Beam search evaluation cho model tốt nhất
!python src/evaluate.py --model_type D --checkpoint checkpoints/model_d_best.pth --beam_width 5 --no_repeat_ngram 3

---
## 10. Phase 3: Full Optimization (+5 epochs)

Phase 3 kích hoạt toàn bộ cải tiến:
- **GloVe 300d Embeddings**: pretrained word vectors cho cả Q-encoder và decoder
- **Coverage Mechanism** (Model C/D): phạt khi attend lại vùng đã attend
- **Gradient Accumulation**: effective batch = batch_size × accum_steps
- Continue scheduled sampling + augmentation + CNN fine-tuning

In [ ]:
# ═══ PHASE 3: Model A (+5 epochs, GloVe + grad accum) ═══
!python src/train.py \
    --model A \
    --epochs 5 \
    --lr 2e-4 \
    --batch_size 128 \
    --scheduled_sampling \
    --augment \
    --glove --glove_dim 300 \
    --accum_steps 2 \
    --early_stopping 3 \
    --resume checkpoints/model_a_resume.pth

In [ ]:
# ═══ PHASE 3: Model B (+5 epochs, GloVe + finetune + grad accum) ═══
!python src/train.py \
    --model B \
    --epochs 5 \
    --lr 2e-4 \
    --batch_size 128 \
    --scheduled_sampling \
    --finetune_cnn \
    --cnn_lr_factor 0.1 \
    --augment \
    --glove --glove_dim 300 \
    --accum_steps 2 \
    --early_stopping 3 \
    --resume checkpoints/model_b_resume.pth

In [ ]:
# ═══ PHASE 3: Model C (+5 epochs, GloVe + coverage + grad accum) ═══
!python src/train.py \
    --model C \
    --epochs 5 \
    --lr 2e-4 \
    --batch_size 64 \
    --scheduled_sampling \
    --augment \
    --glove --glove_dim 300 \
    --coverage --coverage_lambda 1.0 \
    --accum_steps 4 \
    --early_stopping 3 \
    --resume checkpoints/model_c_resume.pth

In [ ]:
# ═══ PHASE 3: Model D (+5 epochs, FULL — GloVe + coverage + finetune + grad accum) ═══
!python src/train.py \
    --model D \
    --epochs 5 \
    --lr 2e-4 \
    --batch_size 64 \
    --scheduled_sampling \
    --finetune_cnn \
    --cnn_lr_factor 0.05 \
    --augment \
    --glove --glove_dim 300 \
    --coverage --coverage_lambda 1.0 \
    --accum_steps 4 \
    --early_stopping 3 \
    --resume checkpoints/model_d_resume.pth

---
## 11. Phase 3: Final Evaluation

### 11.1 Full Training Curves

In [ ]:
# Complete training curves (all 3 phases)
import json, os
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'A': '#1f77b4', 'B': '#ff7f0e', 'C': '#2ca02c', 'D': '#d62728'}

for model_type in ['A', 'B', 'C', 'D']:
    history_path = f'checkpoints/history_model_{model_type.lower()}.json'
    if os.path.exists(history_path):
        with open(history_path) as f:
            history = json.load(f)
        epochs = range(1, len(history['train_loss']) + 1)
        axes[0].plot(epochs, history['train_loss'], label=f'Model {model_type}',
                     color=colors[model_type], linewidth=2)
        axes[1].plot(epochs, history['val_loss'], label=f'Model {model_type}',
                     color=colors[model_type], linewidth=2)

# Phase boundaries
for ax in axes:
    ax.axvline(10, color='gray', linestyle='--', alpha=0.4, label='Phase 1→2')
    ax.axvline(15, color='gray', linestyle=':', alpha=0.4, label='Phase 2→3')
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[1].set_title('Validation Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')

plt.suptitle('Complete Training: Phase 1 → 2 → 3', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/final_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.2 Final Comparison (Greedy + Beam Search)

In [ ]:
# Greedy comparison (fast)
!python src/compare.py --epoch 20 --models A,B,C,D

In [ ]:
# Beam search comparison (more accurate but slower)
!python src/compare.py --epoch 20 --models A,B,C,D --beam_width 5 --no_repeat_ngram 3

### 11.3 Detailed Final Evaluation (Best Model)

In [ ]:
# Evaluate best model (D) in detail
!python src/evaluate.py \
    --model_type D \
    --checkpoint checkpoints/model_d_best.pth \
    --beam_width 5 \
    --no_repeat_ngram 3

### 11.4 Phase-by-Phase Comparison Chart

In [ ]:
# So sánh kết quả qua 3 phases (cần chạy compare.py trước)
# Tạo biểu đồ so sánh thủ công từ kết quả
# Điền kết quả thực tế sau khi chạy compare.py

# Ví dụ template (thay bằng kết quả thực):
import matplotlib.pyplot as plt
import numpy as np

models = ['A', 'B', 'C', 'D']
# Placeholder — điền kết quả thực sau khi train
phase1_bleu4 = [0.0, 0.0, 0.0, 0.0]  # epoch 10
phase2_bleu4 = [0.0, 0.0, 0.0, 0.0]  # epoch 15
phase3_bleu4 = [0.0, 0.0, 0.0, 0.0]  # epoch 20

x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width, phase1_bleu4, width, label='Phase 1 (epoch 10)', color='#1f77b4')
bars2 = ax.bar(x, phase2_bleu4, width, label='Phase 2 (epoch 15)', color='#ff7f0e')
bars3 = ax.bar(x + width, phase3_bleu4, width, label='Phase 3 (epoch 20)', color='#2ca02c')

ax.set_ylabel('BLEU-4 Score')
ax.set_title('BLEU-4 Improvement Across Training Phases')
ax.set_xticks(x)
ax.set_xticklabels([f'Model {m}' for m in models])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/phase_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("⚠️ Placeholder data — update with actual results after training!")

---
## 12. Attention Visualization

Chỉ áp dụng cho Model C và D (có Bahdanau Attention).

Tại mỗi decode step, attention weights `alpha (49,)` cho biết model đang "nhìn" vào vùng nào của ảnh (7×7 grid). 

Overlay attention heatmap lên ảnh gốc để trực quan hóa.

In [ ]:
# Visualize attention cho Model D
!python src/visualize.py --model_type D --epoch 20 --sample_idx 0 --output results/attn_model_d_sample0.png
!python src/visualize.py --model_type D --epoch 20 --sample_idx 42 --output results/attn_model_d_sample42.png
!python src/visualize.py --model_type D --epoch 20 --sample_idx 100 --output results/attn_model_d_sample100.png

In [ ]:
# Visualize attention cho Model C
!python src/visualize.py --model_type C --epoch 20 --sample_idx 0 --output results/attn_model_c_sample0.png

In [ ]:
# Hiển thị kết quả attention visualization
import matplotlib.pyplot as plt
from PIL import Image
import os

fig, axes = plt.subplots(2, 1, figsize=(20, 8))

for i, (model, idx) in enumerate([('d', 0), ('c', 0)]):
    path = f'results/attn_model_{model}_sample{idx}.png'
    if os.path.exists(path):
        img = Image.open(path)
        axes[i].imshow(img)
        axes[i].set_title(f'Model {model.upper()} — Sample {idx}', fontsize=12)
    else:
        axes[i].text(0.5, 0.5, f'File not found:\n{path}', ha='center', va='center')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

### 12.1 Custom Attention Visualization (Interactive)

In [ ]:
# Tự chọn sample để visualize
import sys, torch
sys.path.append('src')
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from vocab import Vocabulary
from inference import get_model

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab_q = Vocabulary(); vocab_q.load('data/processed/vocab_questions.json')
vocab_a = Vocabulary(); vocab_a.load('data/processed/vocab_answers.json')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def visualize_sample(model_type, checkpoint, img_path, question, max_tokens=15):
    """Visualize attention for a single sample."""
    model = get_model(model_type, len(vocab_q), len(vocab_a))
    model.load_state_dict(torch.load(checkpoint, map_location='cpu'))
    model.to(DEVICE).eval()
    
    original = Image.open(img_path).convert('RGB')
    img_tensor = transform(original)
    q_tensor = torch.tensor(vocab_q.numericalize(question), dtype=torch.long)
    
    # Decode step by step
    with torch.no_grad():
        img = img_tensor.unsqueeze(0).to(DEVICE)
        q = q_tensor.unsqueeze(0).to(DEVICE)
        
        img_features = F.normalize(model.i_encoder(img), p=2, dim=-1)
        q_feat, q_hidden = model.q_encoder(q)
        fusion = model.fusion(img_features.mean(dim=1), q_feat)
        
        h0 = fusion.unsqueeze(0).repeat(model.num_layers, 1, 1)
        c0 = torch.zeros_like(h0)
        hidden = (h0, c0)
        
        token = torch.tensor([[1]], dtype=torch.long, device=DEVICE)  # <start>
        tokens, alphas = [], []
        coverage = None
        
        for _ in range(max_tokens):
            logit, hidden, alpha, coverage = model.decoder.decode_step(
                token, hidden, img_features, q_hidden, coverage
            )
            pred = logit.argmax(dim=-1).item()
            if pred == 2:  # <end>
                break
            tokens.append(vocab_a.idx2word.get(pred, '<unk>'))
            alphas.append(alpha.squeeze(0).cpu().numpy())
            token = torch.tensor([[pred]], dtype=torch.long, device=DEVICE)
    
    # Plot
    n = len(tokens) + 1
    fig, axes = plt.subplots(1, min(n, 12), figsize=(3 * min(n, 12), 3.5))
    if n == 1:
        axes = [axes]
    
    # Denormalize image
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img_np = (img_tensor * std + mean).clamp(0,1).permute(1,2,0).numpy()
    
    axes[0].imshow(img_np); axes[0].set_title('Original'); axes[0].axis('off')
    
    for i, (word, alpha) in enumerate(zip(tokens[:11], alphas[:11])):
        attn = alpha.reshape(7,7)
        attn = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)
        attn_up = np.array(Image.fromarray((attn*255).astype(np.uint8)).resize((224,224), Image.BILINEAR)) / 255.0
        axes[i+1].imshow(img_np)
        axes[i+1].imshow(attn_up, alpha=0.5, cmap='jet')
        axes[i+1].set_title(f'"{word}"', fontsize=9)
        axes[i+1].axis('off')
    
    answer = ' '.join(tokens)
    fig.suptitle(f'Q: {question}\nA: {answer}', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    del model
    return answer

# Ví dụ sử dụng:
# answer = visualize_sample('D', 'checkpoints/model_d_best.pth',
#                           'data/raw/images/val2014/COCO_val2014_000000000073.jpg',
#                           'Is this a motorcycle or a bike?')

---
## 13. Interactive Inference Demo

### 13.1 Single Sample Inference

In [ ]:
# Single sample inference
import sys, torch, json
sys.path.append('src')
from PIL import Image
from torchvision import transforms
from vocab import Vocabulary
from inference import (get_model, greedy_decode, greedy_decode_with_attention,
                       beam_search_decode, beam_search_decode_with_attention)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab_q = Vocabulary(); vocab_q.load('data/processed/vocab_questions.json')
vocab_a = Vocabulary(); vocab_a.load('data/processed/vocab_answers.json')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict(model_type, checkpoint, img_path, question, beam_width=1):
    """Run inference with greedy or beam search decode."""
    model = get_model(model_type, len(vocab_q), len(vocab_a))
    model.load_state_dict(torch.load(checkpoint, map_location='cpu'))
    model.to(DEVICE).eval()
    
    img_tensor = transform(Image.open(img_path).convert('RGB'))
    q_tensor = torch.tensor(vocab_q.numericalize(question), dtype=torch.long)
    
    use_attn = model_type in ('C', 'D')
    
    if beam_width > 1:
        if use_attn:
            answer = beam_search_decode_with_attention(
                model, img_tensor, q_tensor, vocab_a,
                beam_width=beam_width, device=DEVICE, no_repeat_ngram_size=3
            )
        else:
            answer = beam_search_decode(
                model, img_tensor, q_tensor, vocab_a,
                beam_width=beam_width, device=DEVICE, no_repeat_ngram_size=3
            )
    else:
        if use_attn:
            answer = greedy_decode_with_attention(model, img_tensor, q_tensor, vocab_a, device=DEVICE)
        else:
            answer = greedy_decode(model, img_tensor, q_tensor, vocab_a, device=DEVICE)
    
    del model
    return answer

# Ví dụ:
# answer = predict('D', 'checkpoints/model_d_best.pth',
#                  'data/raw/images/val2014/COCO_val2014_000000000073.jpg',
#                  'Is this a motorcycle or a bike?',
#                  beam_width=5)
# print(f"Answer: {answer}")

### 13.2 Batch Inference trên Val Set

In [ ]:
# Batch inference demo — compare greedy vs beam search
import random

with open('data/raw/vqa_e_json/VQA-E_val_set.json') as f:
    val_data = json.load(f)

# Chọn 10 samples ngẫu nhiên
test_samples = random.sample(val_data, 10)
model_type = 'D'  # thay đổi model ở đây
ckpt = f'checkpoints/model_{model_type.lower()}_best.pth'

if os.path.exists(ckpt):
    model = get_model(model_type, len(vocab_q), len(vocab_a))
    model.load_state_dict(torch.load(ckpt, map_location='cpu'))
    model.to(DEVICE).eval()
    
    print(f"{'='*80}")
    print(f"Model {model_type} — Greedy vs Beam Search (width=5)")
    print(f"{'='*80}")
    
    for s in test_samples:
        img_path = f"data/raw/images/val2014/COCO_val2014_{s['img_id']:012d}.jpg"
        if not os.path.exists(img_path):
            continue
        
        img_tensor = transform(Image.open(img_path).convert('RGB'))
        q_tensor = torch.tensor(vocab_q.numericalize(s['question']), dtype=torch.long)
        
        # Greedy
        greedy = greedy_decode_with_attention(model, img_tensor, q_tensor, vocab_a, device=DEVICE) \
                 if model_type in ('C', 'D') else \
                 greedy_decode(model, img_tensor, q_tensor, vocab_a, device=DEVICE)
        
        # Beam search
        beam = beam_search_decode_with_attention(
            model, img_tensor, q_tensor, vocab_a, beam_width=5, device=DEVICE, no_repeat_ngram_size=3
        ) if model_type in ('C', 'D') else \
        beam_search_decode(
            model, img_tensor, q_tensor, vocab_a, beam_width=5, device=DEVICE, no_repeat_ngram_size=3
        )
        
        gt = f"{s['multiple_choice_answer']} because {s.get('explanation', [''])[0][:80]}"
        print(f"Q: {s['question']}")
        print(f"  GT    : {gt}...")
        print(f"  Greedy: {greedy}")
        print(f"  Beam-5: {beam}")
        print()
    
    del model
else:
    print(f"Checkpoint not found: {ckpt}")
    print("Please train the model first.")

### 13.3 Upload ảnh tùy chọn & hỏi

In [ ]:
# Upload ảnh từ local machine (Colab)
from google.colab import files
import io

print("Upload an image to ask a question about it:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    img = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')
    
    # Hiển thị ảnh
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Uploaded Image')
    plt.show()
    
    # Lưu tạm thời
    temp_path = '/tmp/uploaded_image.jpg'
    img.save(temp_path)
    
    # Hỏi câu hỏi
    question = input("Enter your question: ")
    
    # Predict
    for model_type in ['A', 'D']:
        ckpt = f'checkpoints/model_{model_type.lower()}_best.pth'
        if os.path.exists(ckpt):
            answer = predict(model_type, ckpt, temp_path, question, beam_width=5)
            print(f"  Model {model_type}: {answer}")

---
## 14. Save Results & Backup

### 14.1 Backup checkpoints to Google Drive

In [ ]:
# Backup checkpoints và results vào Google Drive
import shutil, os

DRIVE_BACKUP = '/content/drive/MyDrive/vqa_new_results'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
os.makedirs(f'{DRIVE_BACKUP}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_BACKUP}/results', exist_ok=True)

# Copy checkpoints
for f in os.listdir('checkpoints'):
    if f.endswith('.pth') or f.endswith('.json'):
        src = f'checkpoints/{f}'
        dst = f'{DRIVE_BACKUP}/checkpoints/{f}'
        shutil.copy2(src, dst)
        print(f"  Copied: {f}")

# Copy results
if os.path.exists('results'):
    for f in os.listdir('results'):
        shutil.copy2(f'results/{f}', f'{DRIVE_BACKUP}/results/{f}')
        print(f"  Copied: results/{f}")

print(f"\nAll files backed up to: {DRIVE_BACKUP}")

### 14.2 Tổng kết kết quả

In [ ]:
# Tổng kết tất cả history
import json, os

print("=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)

for model_type in ['A', 'B', 'C', 'D']:
    path = f'checkpoints/history_model_{model_type.lower()}.json'
    if os.path.exists(path):
        with open(path) as f:
            h = json.load(f)
        n_epochs = len(h['train_loss'])
        best_val = min(h['val_loss'])
        best_epoch = h['val_loss'].index(best_val) + 1
        final_train = h['train_loss'][-1]
        final_val = h['val_loss'][-1]
        print(f"\nModel {model_type}:")
        print(f"  Total epochs: {n_epochs}")
        print(f"  Best val loss: {best_val:.4f} (epoch {best_epoch})")
        print(f"  Final train/val: {final_train:.4f} / {final_val:.4f}")
    else:
        print(f"\nModel {model_type}: No history found")

print("\n" + "=" * 70)
print("Run compare.py with --epoch 20 for a final BLEU-4 / METEOR / BERTScore comparison")
print("=" * 70)

---
## 📝 Ghi chú

### Hyperparameters Summary
| Parameter | Phase 1 | Phase 2 | Phase 3 |
|-----------|---------|---------|---------|
| Epochs | 10 | +5 | +5 |
| Learning Rate | 1e-3 | 5e-4 | 2e-4 |
| Batch Size (A/B) | 128 | 128 | 128 |
| Batch Size (C/D) | 64 | 64 | 64 |
| Scheduled Sampling | ❌ | ✅ (k=5) | ✅ (k=5) |
| CNN Fine-tune (B/D) | ❌ | ✅ (lr×0.1) | ✅ (lr×0.05) |
| Data Augmentation | ❌ | ✅ | ✅ |
| GloVe Embeddings | ❌ | ❌ | ✅ (300d) |
| Coverage (C/D) | ❌ | ❌ | ✅ (λ=1.0) |
| Grad Accumulation | ❌ | ❌ | ✅ (×2 or ×4) |
| Early Stopping | ❌ | 3 epochs | 3 epochs |

### CLI Reference
```bash
# Full training command (Phase 3, Model D):
python src/train.py \
    --model D --epochs 5 --lr 2e-4 --batch_size 64 \
    --scheduled_sampling --finetune_cnn --cnn_lr_factor 0.05 \
    --augment --glove --glove_dim 300 \
    --coverage --coverage_lambda 1.0 --accum_steps 4 \
    --early_stopping 3 --resume checkpoints/model_d_resume.pth

# Evaluation:
python src/evaluate.py --model_type D --checkpoint checkpoints/model_d_best.pth --beam_width 5

# Compare all models:
python src/compare.py --epoch 20 --beam_width 5 --no_repeat_ngram 3

# Visualize attention:
python src/visualize.py --model_type D --epoch 20 --sample_idx 42
```